# Data statistics

Owner: Tiến. Every number is computed at build time from `data/m1/papers_stats.csv` and `data/m1/references_stats.csv`.

A **successful paper** is a paper whose versions all reached a final status (`n_versions_ok == n_versions`), see [Methodology](01-methodology.md).

In [ ]:
from pathlib import Path

import matplotlib.pyplot as plt
import pandas as pd

DATA = Path("../data/m1")
FIG = Path("../figures")
FIG.mkdir(exist_ok=True)

df = pd.read_csv(DATA / "papers_stats.csv", dtype={"arxiv_id": str})
ok = df.n_versions_ok == df.n_versions

## Overview

In [ ]:
summary = {
    "Papers in the ranges": len(df),
    "Papers crawled successfully": int(ok.sum()),
    "Success rate (%)": round(100 * ok.mean(), 2),
    "Mean size before removing figures (MB)": round(df.size_before_bytes[ok].mean() / 1e6, 2),
    "Mean size after removing figures (MB)": round(df.size_after_bytes[ok].mean() / 1e6, 3),
    "Mean references per paper": round(df.n_refs_total.mean(), 1),
    "References with resolved metadata (%)": round(
        100 * df.n_refs_meta_ok.sum() / df.n_refs_total.sum(), 2
    ),
    "Mean processing time per paper (s)": round(df.process_sec.mean(), 2),
}
if "ref_status" in df:  # present in the real export (src/analysis/stats.py)
    summary["Reference metadata success rate (% of papers)"] = round(
        100 * (df.ref_status == "ok").mean(), 2
    )
pd.Series(summary).to_frame("Value")

## Size before and after removing figures

In [ ]:
fig, ax = plt.subplots(figsize=(6, 3.5))
ax.hist(
    [df.size_before_bytes[ok] / 1e6, df.size_after_bytes[ok] / 1e6],
    bins=50,
    log=True,
    label=["Before", "After"],
)
ax.set_xlabel("MB per paper")
ax.set_ylabel("Papers (log scale)")
ax.legend()
fig.tight_layout()
fig.savefig(FIG / "m1_size_hist.svg")  # vector copy reused on the poster

## Versions and references

In [ ]:
df.n_versions.value_counts().sort_index().rename_axis("Versions").to_frame("Papers")

In [ ]:
fig, ax = plt.subplots(figsize=(6, 3.5))
ax.hist(df.n_refs_total, bins=40)
ax.set_xlabel("References per paper")
ax.set_ylabel("Papers")
fig.tight_layout()
fig.savefig(FIG / "m1_refs_hist.svg")

## Reference metadata by source

In [ ]:
refs = pd.read_csv(DATA / "references_stats.csv")
refs["hit_rate_%"] = (100 * refs.n_hit / (refs.n_hit + refs.n_miss)).round(2)
refs